# 📝 Natural Language Processing (NLP) Foundations & Text Preprocessing

Natural Language Processing (NLP) enables computational systems to process, analyze, and extract meaningful representations from human language. Because machine learning models require structured numerical inputs, unstructured raw text must undergo rigorous linguistic preprocessing and numerical vectorization.

This practical guide covers:
- **NLTK Setup & Corpus Resources**: Tokenizers, Stopwords, WordNet
- **Text Tokenization**: Word-level and sentence-level segmentation
- **Stopword Filtering**: Identifying and purging non-informative functional words
- **Morphological Normalization**: Porter Stemmer, Snowball Stemmer vs. WordNet Lemmatization
- **Regex Cleaning**: Noise filtering, symbol purging, and whitespace normalization
- **End-to-End Pipeline**: A production-ready text cleaning utility function
- **Numerical Vectorization**: Bag-of-Words (`CountVectorizer`) and TF-IDF (`TfidfVectorizer`)


### 1. Library Imports
Import `nltk`, regular expressions (`re`), `string`, `pandas`, `numpy`, and text vectorizers from `scikit-learn`.


In [ ]:
import re
import string
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import warnings
warnings.filterwarnings('ignore')


### 2. Downloading NLTK Corpora & Tokenizer Models
Download required linguistic models:
- `stopwords`: High-frequency English functional words
- `punkt` & `punkt_tab`: Unsupervised sentence boundary and word tokenizers
- `wordnet` & `omw-1.4`: Lexical database for lemmatization


In [ ]:
for resource in ['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Error downloading {resource}: {e}")

print("NLTK linguistic resources verified and loaded successfully.")


### 3. Inspecting Stopwords
Stop words are high-frequency grammatical words (articles, conjunctions, prepositions) that carry little semantic discriminative power in text classification tasks. Load the standard English stopword lexicon.


In [ ]:
stop_words = stopwords.words('english')

print(f"Total English Stopwords: {len(stop_words)}")
print("\nSample Stopwords (first 25):")
print(stop_words[:25])


### 4. Text Tokenization: Word and Sentence Segmentation
Tokenization segments raw text into discrete tokens.
- **Word Tokenization**: Splits running text into constituent words and punctuation.
- **Sentence Tokenization**: Splits multi-sentence passages using punctuation and capitalization rules.


In [ ]:
sample_text = "The movie was extraordinarily good! Have you seen it yet? I'd highly recommend it."

# Sentence Tokenization
sentences = sent_tokenize(sample_text)
print("Sentence Tokenization:")
for idx, s in enumerate(sentences):
    print(f"  [{idx}] {s}")

# Word Tokenization
words = word_tokenize(sample_text)
print(f"\nWord Tokenization ({len(words)} tokens):")
print(words)


### 5. Case Normalization & Stopword Removal
Standardize text to lowercase (case-folding) and filter out tokens matching the stopword list to retain content-bearing words.


In [ ]:
review = '''The Natural Language Toolkit (NLTK) is an open source Python library
for Natural Language Processing. A free online book is available.'''

tokens = word_tokenize(review)

# Lowercase and filter non-alphabetical or stopword tokens
filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]

print(f"Original Token Count: {len(tokens)}")
print(f"Filtered Token Count: {len(filtered_tokens)}")
print("\nFiltered Tokens:")
print(filtered_tokens)


### 6. Morphological Normalization: Stemming
Stemming uses heuristic suffix-stripping rules to truncate inflected words down to a common root stem.
- **Porter Stemmer**: The foundational Martin Porter algorithm (1980).
- **Snowball Stemmer**: An optimized, more aggressive English stemmer (Porter2).
Note: Stems are not guaranteed to be valid dictionary words (e.g. `'programms'` $\rightarrow$ `'programm'`).


In [ ]:
sample_words = ['programms', 'Programming', 'reached', 'walking', 'jumping', 'eating', 'rocks', 'reaches']

porter = PorterStemmer()
snowball = SnowballStemmer('english')

comparison_data = []
for word in sample_words:
    comparison_data.append({
        'Original Word': word,
        'Porter Stemmer': porter.stem(word),
        'Snowball Stemmer': snowball.stem(word)
    })

stem_df = pd.DataFrame(comparison_data)
stem_df


### 7. Morphological Normalization: Lemmatization
Lemmatization leverages the WordNet lexical database to resolve words to their canonical dictionary headword (lemma).
- Unlike stemming, lemmatization always produces an authentic, grammatically correct word.
- By providing Part-of-Speech (**POS**) tags (`'n'` for noun, `'v'` for verb, `'a'` for adjective), the lemmatizer accurately disambiguates inflections.


In [ ]:
lemmatizer = WordNetLemmatizer()

lem_results = []
for word in sample_words:
    lem_results.append({
        'Original Word': word,
        'Default (Noun)': lemmatizer.lemmatize(word),
        'Verb POS (pos="v")': lemmatizer.lemmatize(word.lower(), pos='v')
    })

lem_df = pd.DataFrame(lem_results)
lem_df


### 8. Text Cleaning with Regular Expressions (`re`)
Raw real-world text often contains markup artifacts, special characters, noisy punctuation, and irregular spacing.
Use regex pattern substitution `re.sub(r'[^a-zA-Z0-9]', ' ', text)` to preserve alphanumeric content and collapse redundant whitespaces.


In [ ]:
noisy_string = "L^&*uminar te#$%chn5853olab @# is in$34 Ka!@$%kkan!@ad"

# Replace non-alphanumeric symbols with single spaces
cleaned_step1 = re.sub(r'[^a-zA-Z0-9]', ' ', noisy_string)

# Normalize consecutive whitespaces into a single space
cleaned_step2 = re.sub(r'\s+', ' ', cleaned_step1).strip()

print("Original String :", noisy_string)
print("After Regex Sub :", cleaned_step1)
print("Normalized Spacing:", cleaned_step2)


### 9. End-to-End Text Preprocessing Pipeline
Synthesize regex cleaning, case-folding, tokenization, stopword removal, and WordNet lemmatization into a reusable function.


In [ ]:
def clean_and_preprocess_text(text):
    # 1. Remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # 2. Case-folding to lowercase
    text = text.lower()
    # 3. Tokenization
    tokens = word_tokenize(text)
    # 4. Stopword filtering & Lemmatization
    lemmatizer = WordNetLemmatizer()
    stop_set = set(stopwords.words('english'))
    cleaned_tokens = [
        lemmatizer.lemmatize(token, pos='v')
        for token in tokens
        if token not in stop_set and len(token) > 2
    ]
    return ' '.join(cleaned_tokens)

# Demonstration on sample documents
sample_corpus = [
    "Machine Learning algorithms learn meaningful patterns directly from labeled training datasets!",
    "Natural Language Processing enables computers to understand, analyze, and generate human languages.",
    "Deep learning models with neural networks are advancing image recognition and translation systems."
]

cleaned_corpus = [clean_and_preprocess_text(doc) for doc in sample_corpus]

for original, cleaned in zip(sample_corpus, cleaned_corpus):
    print(f"Original: {original}")
    print(f"Cleaned : {cleaned}\n")


### 10. Text Vectorization: Bag-of-Words (CountVectorizer)
Bag-of-Words (BoW) constructs a fixed vocabulary of all unique tokens across the corpus and transforms each document into a vector representing token occurrence counts.


In [ ]:
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(cleaned_corpus)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f'Doc_{i+1}' for i in range(len(cleaned_corpus))]
)

print(f"Vocabulary Size: {len(count_vectorizer.get_feature_names_out())} unique terms")
bow_df


### 11. Text Vectorization: TF-IDF (Term Frequency - Inverse Document Frequency)
TF-IDF balances raw term frequency against corpus-wide inverse document frequency:
$$TF\text{-}IDF(t, d) = TF(t, d) \times \left(\log\frac{1 + |D|}{1 + DF(t)} + 1\right)$$
Words that appear across every document receive lower weights, while words distinctive to specific documents receive higher weights.


In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_corpus)

tfidf_df = pd.DataFrame(
    np.round(tfidf_matrix.toarray(), 3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Doc_{i+1}' for i in range(len(cleaned_corpus))]
)

tfidf_df


### 12. Key Takeaways & NLP Workflow Best Practices
1. **Stemming vs. Lemmatization**: Use stemming when processing speed is critical (e.g., search indexing); use lemmatization when grammatical validity and lexical nuance matter (e.g., medical NLP, sentiment analysis).
2. **Stopwords Strategy**: Removing stopwords helps reduce feature space dimensionality for topic modeling and naive text classification, but should be retained for syntactic parsing, translation, and large language models (LLMs).
3. **BoW vs. TF-IDF**: TF-IDF systematically outperforms raw count vectors by penalizing common corpus terms and elevating discriminating keywords for downstream classifiers (e.g., Naive Bayes, Logistic Regression, Support Vector Machines).
